In [87]:
import gc

gc.collect()

46415

# elevator-rmnd-lstm

Use the simulated dataset generated by `dataset.py` to train
regression models, in order to predict the remaining useful life (RUL)
of lifts.

**This approach uses an LSTM.**

In [88]:
from datetime import datetime
# import dataset
import numpy as np
import pandas as pd
import seaborn as sns
import os

!pip install wandb weave pytorch_lightning torchmetrics
!wandb login --relogin

sns.set_theme(context="notebook", style="ticks")

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


## preprocessing
* Import the simulated dataset
* ~~Convert lift ids to integers~~ [already handled by `dataset.py`]
* ~~Compute instantaneous time derivatives of each sensor metric~~ [already handled by `dataset.py`]
* ~~Compute RUL in h~~ [already handled by `dataset.py`]
* Split data into training/testing sets
* Scale data with a `Scaler`

In [ ]:
from google.colab import files
uploaded = files.upload()

FNAME = "liftdata_v3.pkl"
df_full = pd.read_pickle(FNAME)
df_full.head(10)

,timestamp,lift_id,lift_model,lift_age_hours,ARM_DIST_mm,DOOR_DIST_mm,FLOOR_DIST_mm,ROPE_MFL_mV,BEARING_TEMP_C,maintenance_done,...,arm_door_ratio,floor_door_ratio,temp_x_rope,cumulative_rope_degradation,cumulative_bearing_heat,ARM_DIST_mm_per_hr,DOOR_DIST_mm_per_hr,FLOOR_DIST_mm_per_hr,ROPE_MFL_mV_per_hr,BEARING_TEMP_C_per_hr
0,2023-01-01 00:00:00,1,Otis Gen2,15000,0.480,1.996,0.028,10.045,33.693,0,...,0.240481,0.014028,338.446185,10.045,33.693,0.0000,0.0000,0.0000,0.0000,0.0000
1,2023-01-01 12:00:00,1,Otis Gen2,15012,0.527,2.065,0.144,9.681,36.860,0,...,0.255206,0.069734,356.841660,19.726,70.553,0.0039,0.0057,0.0097,-0.0303,0.2639
2,2023-01-02 00:00:00,1,Otis Gen2,15024,0.501,2.244,0.197,9.351,35.014,0,...,0.223262,0.087790,327.415914,29.077,105.567,-0.0022,0.0149,0.0044,-0.0275,-0.1538
3,2023-01-02 12:00:00,1,Otis Gen2,15036,0.498,2.031,0.208,9.458,34.025,0,...,0.245199,0.102413,321.808450,38.535,139.592,-0.0003,-0.0178,0.0009,0.0089,-0.0824
4,2023-01-03 00:00:00,1,Otis Gen2,15048,0.533,2.217,0.471,10.870,36.996,0,...,0.240415,0.212449,402.146520,49.405,176.588,0.0029,0.0155,0.0219,0.1177,0.2476
5,2023-01-03 12:00:00,1,Otis Gen2,15060,0.530,2.276,0.485,9.975,36.493,0,...,0.232865,0.213093,364.017675,59.380,213.081,-0.0003,0.0049,0.0012,-0.0746,-0.0419
6,2023-01-04 00:00:00,1,Otis Gen2,15072,0.536,2.389,0.085,9.975,36.464,0,...,0.224362,0.035580,363.728400,69.355,249.545,0.0005,0.0094,-0.0333,0.0000,-0.0024
7,2023-01-04 12:00:00,1,Otis Gen2,15084,0.556,2.166,0.446,10.252,36.737,0,...,0.256694,0.205909,376.627724,79.607,286.282,0.0017,-0.0186,0.0301,0.0231,0.0228
8,2023-01-05 00:00:00,1,Otis Gen2,15096,0.585,2.249,0.132,10.477,37.251,0,...,0.260115,0.058693,390.278727,90.084,323.533,0.0024,0.0069,-0.0262,0.0187,0.0428
9,2023-01-05 12:00:00,1,Otis Gen2,15108,0.573,2.364,0.834,11.572,36.114,0,...,0.242386,0.352792,417.911208,101.656,359.647,-0.0010,0.0096,0.0585,0.0912,-0.0948


In [105]:
# Remove unnecessary columns
df_full = df_full.drop(columns=["timestamp", "lift_model", "maintenance_done"])

# Then remove NaN values
df_full.dropna(inplace=True)

Since there exist rows that have `np.inf` as their predicted RUL, extract those rows for experimentation later on.
Retain the remaining roles (with valid RULs) for training and testing.

In [106]:
df_experimental = df_full[df_full["RUL_hrs"] == np.inf]
df_useful = df_full[df_full["RUL_hrs"] != np.inf]

# Check to see the experimental/useful split was done correctly
assert all(
    df_experimental["RUL_hrs"] == np.inf
), "All records in experimental set should have RUL_hrs == np.inf"
assert all(
    df_useful["RUL_hrs"] != np.inf
), "All records in useful set should have valid RUL_hrs"

df_useful.head(40)

,lift_id,lift_age_hours,ARM_DIST_mm,DOOR_DIST_mm,FLOOR_DIST_mm,ROPE_MFL_mV,BEARING_TEMP_C,RUL_hrs,age_since_last_maint,arm_door_ratio,floor_door_ratio,temp_x_rope,cumulative_rope_degradation,cumulative_bearing_heat,ARM_DIST_mm_per_hr,DOOR_DIST_mm_per_hr,FLOOR_DIST_mm_per_hr,ROPE_MFL_mV_per_hr,BEARING_TEMP_C_per_hr
142,1,16704,1.507,7.714,10.128,17.462,75.193,0.0,0.000000,0.195359,1.312937,1313.020166,2059.949,7925.487,0.0027,0.0016,0.0021,-0.1110,-0.0293
143,1,16716,0.492,2.087,0.079,9.907,33.291,1680.0,0.017241,0.235745,0.037853,329.813937,2069.856,7958.778,-0.0846,-0.4689,-0.8374,-0.6296,-3.4918
144,1,16728,0.483,2.071,0.005,10.555,34.098,1668.0,0.034483,0.233221,0.002414,359.904390,2080.411,7992.876,-0.0008,-0.0013,-0.0062,0.0540,0.0673
145,1,16740,0.540,2.087,0.322,10.248,35.311,1656.0,0.051724,0.258744,0.154288,361.867128,2090.659,8028.187,0.0048,0.0013,0.0264,-0.0256,0.1011
146,1,16752,0.528,2.177,0.241,10.670,34.296,1644.0,0.068966,0.242535,0.110703,365.938320,2101.329,8062.483,-0.0010,0.0075,-0.0068,0.0352,-0.0846
147,1,16764,0.528,2.187,0.183,10.131,35.045,1632.0,0.086207,0.241427,0.083676,355.040895,2111.460,8097.528,0.0000,0.0008,-0.0048,-0.0449,0.0624
148,1,16776,0.504,2.295,0.832,11.048,36.034,1620.0,0.103448,0.219608,0.362527,398.103632,2122.508,8133.562,-0.0020,0.0090,0.0541,0.0764,0.0824
149,1,16788,0.514,2.314,0.669,10.738,36.379,1608.0,0.120690,0.222126,0.289110,390.637702,2133.246,8169.941,0.0008,0.0016,-0.0136,-0.0258,0.0287
150,1,16800,0.502,2.294,0.672,11.244,34.894,1596.0,0.137931,0.218832,0.292938,392.348136,2144.490,8204.835,-0.0010,-0.0017,0.0003,0.0422,-0.1237
151,1,16812,0.538,2.218,0.837,10.575,35.182,1584.0,0.155172,0.242561,0.377367,372.049650,2155.065,8240.017,0.0030,-0.0063,0.0137,-0.0558,0.0240


In [107]:
sensor_cols = [
    'lift_id', 'lift_age_hours', 'ARM_DIST_mm', 'DOOR_DIST_mm',
       'FLOOR_DIST_mm', 'ROPE_MFL_mV', 'BEARING_TEMP_C',
       'age_since_last_maint', 'arm_door_ratio', 'floor_door_ratio',
       'temp_x_rope', 'cumulative_rope_degradation', 'cumulative_bearing_heat',
       'ARM_DIST_mm_per_hr', 'DOOR_DIST_mm_per_hr', 'FLOOR_DIST_mm_per_hr',
       'ROPE_MFL_mV_per_hr', 'BEARING_TEMP_C_per_hr'
]

## sequences for LSTM training

In [108]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

First create train/val/test splits in the ratio 70/15/15.

In [109]:
# Split df_useful into 70/15/15 train/val/test sets, preserving temporal contiguity per lift_id
train_dfs = []
val_dfs = []
test_dfs = []

for lift_id, group in df_useful.groupby('lift_id'):
    # Get the group's data
    group_data = group.reset_index(drop=True)
    n = len(group_data)
    
    # Calculate split indices for contiguous splits
    train_size = int(n * 0.70)
    val_size = int(n * 0.15)
    
    # Split contiguously (preserves temporal order)
    train_dfs.append(group_data[:train_size])
    val_dfs.append(group_data[train_size:train_size + val_size])
    test_dfs.append(group_data[train_size + val_size:])

# Concatenate all splits
train_df = pd.concat(train_dfs, ignore_index=True)
val_df = pd.concat(val_dfs, ignore_index=True)
test_df = pd.concat(test_dfs, ignore_index=True)

train_df.head(20)

,lift_id,lift_age_hours,ARM_DIST_mm,DOOR_DIST_mm,FLOOR_DIST_mm,ROPE_MFL_mV,BEARING_TEMP_C,RUL_hrs,age_since_last_maint,arm_door_ratio,floor_door_ratio,temp_x_rope,cumulative_rope_degradation,cumulative_bearing_heat,ARM_DIST_mm_per_hr,DOOR_DIST_mm_per_hr,FLOOR_DIST_mm_per_hr,ROPE_MFL_mV_per_hr,BEARING_TEMP_C_per_hr
46616,3,87892,0.942,5.194,5.141,16.371,49.671,888.0,1.389831,0.181363,0.989796,813.163941,89661.989,341935.284,0.0007,0.0104,0.0284,0.0392,-0.0435
18902,1,241824,0.795,3.764,1.876,11.621,42.591,924.0,0.655172,0.211211,0.498406,494.950011,254793.273,966039.685,-0.0022,-0.0012,0.0518,-0.0452,-0.0352
32618,2,183416,0.555,2.739,1.107,10.891,36.286,1020.0,0.186441,0.202629,0.404162,395.190826,169406.126,652017.425,-0.0002,0.0104,-0.0049,0.0435,-0.1452
1283,1,30396,0.613,4.268,1.805,13.897,48.604,720.0,0.655172,0.143627,0.422915,675.449788,16841.534,61911.054,-0.0001,-0.0033,0.0102,0.0294,0.0860
24766,2,89192,0.538,2.400,0.749,10.734,40.308,1848.0,0.305085,0.224167,0.312083,432.666072,66361.419,247724.380,-0.0025,-0.0024,0.0025,-0.0124,0.1231
45051,3,69112,1.028,5.558,5.808,12.350,53.514,600.0,1.186441,0.184959,1.044980,660.897900,68284.347,260541.861,-0.0012,0.0205,0.0008,-0.0300,0.2202
27257,2,119084,1.059,5.940,1.924,14.573,41.181,432.0,1.118644,0.178283,0.323906,600.130713,98418.922,375402.132,-0.0013,0.0165,-0.0053,-0.0302,-0.0855
14645,1,190740,1.201,4.828,2.426,16.122,77.552,72.0,1.586207,0.248757,0.502485,1250.293344,195933.558,745350.584,-0.0002,-0.0001,-0.0038,0.0319,0.0547
36576,2,230912,1.105,4.347,8.697,14.358,49.467,732.0,2.033898,0.254198,2.000690,710.247186,222781.842,849139.863,-0.0005,0.0076,0.0377,0.0243,-0.1084
4636,1,70632,0.598,2.288,1.208,10.745,39.448,1272.0,0.258621,0.261364,0.527972,423.868760,61503.784,235072.274,0.0028,-0.0029,-0.0173,-0.0085,-0.1308


Then scale the data with `StandardScaler`.

In [110]:
scaler = StandardScaler()
train_df[sensor_cols] = scaler.fit_transform(train_df[sensor_cols])
val_df[sensor_cols] = scaler.transform(val_df[sensor_cols])
test_df[sensor_cols] = scaler.transform(test_df[sensor_cols])

Lastly, create sequences for training with an RNN.

In [111]:
def create_seqs(data, seq_len, feature_cols, target_col="RUL_hrs"):
    X, y = [], []
    
    # Create sequences within each lift separately
    for lift_id, group in data.groupby('lift_id'):
        features = group[feature_cols].values
        targets = group[target_col].values
        
        for i in range(len(group) - seq_len):
            X.append(features[i : i + seq_len])
            y.append(targets[i + seq_len])
    return np.array(X), np.array(y)


X_train, y_train = create_seqs(train_df, seq_len=50, feature_cols=sensor_cols)
X_val, y_val = create_seqs(val_df, seq_len=50, feature_cols=sensor_cols)
X_test, y_test = create_seqs(test_df, seq_len=50, feature_cols=sensor_cols)

In [ ]:
def create_seqs(data, seq_len, feature_cols, target_col="RUL_hrs"):
    X, y = [], []
    
    # Create sequences within each lift separately
    for lift_id, group in data.groupby('lift_id'):
        features = group[feature_cols].values
        targets = group[target_col].values
        
        for i in range(len(group) - seq_len):
            X.append(features[i : i + seq_len])
            y.append(targets[i + seq_len])
    
    return np.array(X), np.array(y)



This ensures sequences stay within each lift's time series and don't cross lift boundaries.

## build an LSTM!
We use `pytorch_lightning` to build an LSTM and then train it on the datasets created.

In [112]:
!pip install torchmetrics pytorch_lightning weave wandb
import torch
import os
import torchmetrics as tm
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import DataLoader, TensorDataset
import weave
import wandb

In [113]:
# Some important constants
BATCH_SIZE = 32
N_EPOCHS = 25
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
DROPOUT = 0.2
MODEL_DIR = os.path.join(os.getcwd(), "lightning_logs/checkpoints/")

We first create the appropriate datasets and dataloaders, using `TensorDataset` and `DataLoader`.

In [114]:
# create datasets and dataloaders
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)
val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

Define an overarching LSTM class, complete with
- training, validation, and testing steps
- model metrics logged
- initialisation of callbacks and trainer objects

In [115]:
class BasicLSTM(pl.LightningModule):
    """A simple LSTM-based regression model for RUL prediction.
    All subsequent models will use this class as a parent."""
    def __init__(
        self,
        input_size: int = X_train.shape[2],
        output_size: int = 1,
        lr: float = LEARNING_RATE,
        weight_decay: float = WEIGHT_DECAY,
        dropout: float = DROPOUT,
    ):
        super(BasicLSTM, self).__init__()
        self.save_hyperparameters()
        torch.set_float32_matmul_precision("high")

        self.lstm1 = nn.LSTM(input_size, 128, num_layers=1, batch_first=True)
        self.dropout_layer = nn.Dropout(p=self.hparams.dropout)
        self.lstm2 = nn.LSTM(128, 64, num_layers=1, batch_first=True)
        self.fc = nn.Linear(64, output_size)
        self.criterion = nn.MSELoss()

        # assess a few more metrics
        self.mae = tm.MeanAbsoluteError()
        self.r2 = tm.R2Score()
        self.train_mae = self.mae.clone()
        self.val_mae = self.mae.clone()
        self.test_mae = self.mae.clone()
        self.train_r2 = self.r2.clone()
        self.val_r2 = self.r2.clone()
        self.test_r2 = self.r2.clone()

    def forward(self, x):
        out, _ = self.lstm1(x)
        out = self.dropout_layer(out)
        out, _ = self.lstm2(out)
        out = self.fc(out[:, -1, :])
        return out.squeeze(-1)

    def _shared_step(self, batch):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y.float())
        return loss, y, y_hat

    def training_step(self, batch, batch_idx):
        loss, truths, preds = self._shared_step(batch)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        mae = self.train_mae(preds, truths.float())
        r2 = self.train_r2(preds, truths.float())
        self.log("train_mae", mae, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_r2", r2, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, truths, preds = self._shared_step(batch)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        mae = self.val_mae(preds, truths.float())
        r2 = self.val_r2(preds, truths.float())
        self.log("val_mae", mae, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_r2", r2, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        loss, truths, preds = self._shared_step(batch)
        self.log("test_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        mae = self.test_mae(preds, truths.float())
        r2 = self.test_r2(preds, truths.float())
        self.log("test_mae", mae, on_step=False, on_epoch=True, prog_bar=True)
        self.log("test_r2", r2, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def get_callbacks(self, tnow=None, model_name=None):
        early_stop = pl.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, mode="min", verbose=True,
        )
        checkpoint = pl.callbacks.ModelCheckpoint(
            monitor="val_loss",
            dirpath=f"checkpoints/{model_name}_{tnow}",
            filename="{epoch:02d}-{val_loss:.4f}",
            save_top_k=1,
            mode="min",
            verbose=True,
        )
        lr_monitor = pl.callbacks.LearningRateMonitor(logging_interval="epoch")
        return [early_stop, checkpoint, lr_monitor]

    def get_trainer(self, max_epochs: int, callbacks=None, timestamp=None):
        logger = pl.loggers.WandbLogger(project="zeleo")
        return pl.Trainer(
            max_epochs=max_epochs,
            callbacks=callbacks,
            accelerator="auto",
            devices="auto",
            log_every_n_steps=100,
            logger=logger,
            gradient_clip_val=1.0,
            deterministic=False,
            precision="16-mixed" if torch.cuda.is_available() else "32",
        )

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1,
            },
        }

Then we can embark on **training the model**!

In [116]:
model = BasicLSTM()
now = datetime.now().strftime("%Y%m%d-%H%M")
callbacks = model.get_callbacks(tnow=now, model_name="BasicLSTM")
trainer = model.get_trainer(max_epochs=N_EPOCHS, callbacks=callbacks, timestamp=now)
trainer.fit(model, train_loader, val_loader)

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


wandb: Initializing weave.


Output()

weave: Logged in as Weights & Biases user: jiyometrik.
weave: View Weave data at https://wandb.ai/huomi/zeleo/weave
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name          ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ lstm1         │ LSTM              │ 71.7 K │ train │     0 │
│ 1  │ dropout_layer │ Dropout           │      0 │ train │     0 │
│ 2  │ lstm2         │ LSTM              │ 49.7 K │ train │     0 │
│ 3  │ fc            │ Linear            │     65 │ train │     0 │
│ 4  │ criterion     │ MSELoss           │      0 │ train │     0 │
│ 5  │ mae           │ MeanAbsoluteError │      0 │ train │     0 │
│ 6  │ r2            │ R2Score           │      0 │ train │     0 │
│ 7  │ train_mae     │ MeanAbsoluteError │      0 │ train │     0 │
│ 8  │ val_mae       │ MeanAbsoluteError │      0 │ train │     0 │
│ 9  │ test_mae      │ MeanAbsoluteError │      0 │ train │     0 │
│ 10 │ train_r2      │ R2Score           │      0 │ train │     0 │
│ 11 │ val_r2        │ R2Score           │      0 │ train │     0 │
│ 12 │ test_r2       │ R2Score           │      0 │ train │     0 │
└────┴───────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 121 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 121 K                                                                                                
Total estimated model params size (MB): 0.486                                                                      
Modules in train mode: 13                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved. New best score: 652946.875
INFO:pytorch_lightning.utilities.rank_zero:Epoch 0, global step 1297: 'val_loss' reached 652946.87500 (best 652946.87500), saving model to '/content/checkpoints/BasicLSTM_20260611-0324/epoch=00-val_loss=652946.8750.ckpt' as top 1
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 103907.938 >= min_delta = 0.0. New best score: 549038.938
INFO:pytorch_lightning.utilities.rank_zero:Epoch 1, global step 2594: 'val_loss' reached 549038.93750 (best 549038.93750), saving model to '/content/checkpoints/BasicLSTM_20260611-0324/epoch=01-val_loss=549038.9375.ckpt' as top 1
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 89834.594 >= min_delta = 0.0. New best score: 459204.344
INFO:pytorch_lightning.utilities.rank_zero:Epoch 2, global step 3891: 'val_loss' reached 459204.34375 (best 459204.34375), saving model to '/content/checkpoints/BasicLSTM_2

Finally, test the model.

In [117]:
results = trainer.test(model, test_loader, ckpt_path="best")
test_mae = np.round(results[0]["test_mae"], 4)
test_mae_days = np.round(test_mae / 24, 4)

print(f"*** DONE !!! ***")
print(f"*** test_mae = {test_mae} hours = {test_mae_days} days")
wandb.finish()

INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at /content/checkpoints/BasicLSTM_20260611-0324/epoch=18-val_loss=219059.1875.ckpt
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loaded model weights from the checkpoint at /content/checkpoints/BasicLSTM_20260611-0324/epoch=18-val_loss=219059.1875.ckpt


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │       210604.796875       │
│         test_mae          │    381.59979248046875     │
│          test_r2          │   -0.03561634197831154    │
└───────────────────────────┴───────────────────────────┘

*** DONE !!! ***
*** test_mae = 387.065 hours = 16.1277 days


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇██
lr-Adam,███████████████▃▃▃▃▃▃▃▃▁
test_loss,▁
test_mae,▁
test_r2,▁
train_loss,█▆▅▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_mae,█▆▅▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_r2,▁▃▄▅▆▇▇█████████████████
trainer/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
val_loss,█▆▅▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+2,...


View run quiet-pine-7 at: https://wandb.ai/huomi/zeleo/runs/377w8t0i View project at: https://wandb.ai/huomi/zeleo Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)

Find logs at: wandb/run-20260611_032425-377w8t0i/logs

## refine the lstm

Implement another LSTM with a convolutional layer first, as well as an attention mechanism.

In [118]:
class JiyoAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, out):
        weights = torch.softmax(self.attn(out), dim=1)
        ctx = (weights * out).sum(dim=1)
        return ctx, weights


class JiyoLSTM(BasicLSTM):
    """A modified version of the LSTM structure, with convolutional layers and
    an attention mechanism"""

    def __init__(self, ):
        super(JiyoLSTM, self).__init__()#input_size, output_size, lr, dropout, weight_decay)
        self.save_hyperparameters()
        self.drop_p = .25

        # Specify a convolutional layer first
        self.conv = nn.Conv1d(
            in_channels=X_train.shape[2],
            out_channels=32,
            kernel_size=5,
            padding=5//2,
        )
        self.conv_drop = nn.Dropout(p=self.drop_p)

        # Then the remaining LSTM layers
        self.lstm = nn.LSTM(32, 128, num_layers=2, batch_first=True, dropout=self.drop_p)
        self.attention = JiyoAttention(128)

        # and finally the classifier
        self.fc1 = nn.Linear(128, 32)
        self.fc2 = nn.Linear(32, 1)
        self.fc_drop = nn.Dropout(p=self.drop_p)
        self.criterion = nn.MSELoss()


    def forward(self, x):
        # CNN expects multi-dim input
        out = x.permute(0, 2, 1)
        out = self.conv(out)
        out = torch.relu(out)
        out = self.dropout_layer(out)
        out = out.permute(0, 2, 1)

        # LSTM + Attn
        out, _ = self.lstm(out)
        out, attention_weights = self.attention(out)

        # classifier
        out = torch.relu(self.fc1(out))
        out = self.fc2(self.fc_drop(out))

        return out.squeeze(-1)

In [119]:
jiyomodel = JiyoLSTM()
jiyonow = datetime.now().strftime("%Y%m%d-%H%M")
callbacks = jiyomodel.get_callbacks(tnow=jiyonow, model_name="JiyoLSTM")
jiyotrainer = jiyomodel.get_trainer(max_epochs=N_EPOCHS, callbacks=callbacks, timestamp=jiyonow)
jiyotrainer.fit(jiyomodel, train_loader, val_loader)

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Run data is saved locally in wandb/run-20260611_033840-m82slw63

Syncing run splendid-star-8 to Weights & Biases ( docs )

View run at https://wandb.ai/huomi/zeleo/runs/m82slw63

wandb: Initializing weave.


weave: Logged in as Weights & Biases user: jiyometrik.
weave: View Weave data at https://wandb.ai/huomi/zeleo/weave
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name          ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ lstm1         │ LSTM              │ 71.7 K │ train │     0 │
│ 1  │ dropout_layer │ Dropout           │      0 │ train │     0 │
│ 2  │ lstm2         │ LSTM              │ 49.7 K │ train │     0 │
│ 3  │ fc            │ Linear            │     65 │ train │     0 │
│ 4  │ criterion     │ MSELoss           │      0 │ train │     0 │
│ 5  │ mae           │ MeanAbsoluteError │      0 │ train │     0 │
│ 6  │ r2            │ R2Score           │      0 │ train │     0 │
│ 7  │ train_mae     │ MeanAbsoluteError │      0 │ train │     0 │
│ 8  │ val_mae       │ MeanAbsoluteError │      0 │ train │     0 │
│ 9  │ test_mae      │ MeanAbsoluteError │      0 │ train │     0 │
│ 10 │ train_r2      │ R2Score           │      0 │ train │     0 │
│ 11 │ val_r2        │ R2Score           │      0 │ train │     0 │
│ 12 │ test_r2       │ R2Score           │      0 │ train │     0 │
│ 13 │ conv          │ Conv1d            │  1.6 K │ train │     0 │
│ 14 │ conv_drop     │ Dropout           │      0 │ train │     0 │
│ 15 │ lstm          │ LSTM              │  215 K │ train │     0 │
│ 16 │ attention     │ JiyoAttention     │    129 │ train │     0 │
│ 17 │ fc1           │ Linear            │  4.1 K │ train │     0 │
│ 18 │ fc2           │ Linear            │     33 │ train │     0 │
│ 19 │ fc_drop       │ Dropout           │      0 │ train │     0 │
└────┴───────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 342 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 342 K                                                                                                
Total estimated model params size (MB): 1.369                                                                      
Modules in train mode: 21                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved. New best score: 219272.109
INFO:pytorch_lightning.utilities.rank_zero:Epoch 0, global step 1297: 'val_loss' reached 219272.10938 (best 219272.10938), saving model to '/content/checkpoints/JiyoLSTM_20260611-0338/epoch=00-val_loss=219272.1094.ckpt' as top 1
INFO:pytorch_lightning.utilities.rank_zero:Epoch 1, global step 2594: 'val_loss' was not in top 1
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 216.719 >= min_delta = 0.0. New best score: 219055.391
INFO:pytorch_lightning.utilities.rank_zero:Epoch 2, global step 3891: 'val_loss' reached 219055.39062 (best 219055.39062), saving model to '/content/checkpoints/JiyoLSTM_20260611-0338/epoch=02-val_loss=219055.3906.ckpt' as top 1
INFO:pytorch_lightning.utilities.rank_zero:Epoch 3, global step 5188: 'val_loss' was not in top 1
INFO:pytorch_lightning.utilities.rank_zero:Epoch 4, global step 6485: 'val_loss' was not in top 1
INFO:pytorch_lig

In [120]:
jiyoresults = jiyotrainer.test(jiyomodel, test_loader, ckpt_path="best")
jiyotest_mae = np.round(jiyoresults[0]["test_mae"], 4)
jiyotest_mae_days = np.round(jiyotest_mae / 24, 4)

print(f"*** DONE !!! ***")
print(f"*** test_mae = {jiyotest_mae} hours = {jiyotest_mae_days} days")
wandb.finish()

INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at /content/checkpoints/JiyoLSTM_20260611-0338/epoch=02-val_loss=219055.3906.ckpt
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loaded model weights from the checkpoint at /content/checkpoints/JiyoLSTM_20260611-0338/epoch=02-val_loss=219055.3906.ckpt


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │       210624.328125       │
│         test_mae          │    381.88323974609375     │
│          test_r2          │   -0.03623140603303909    │
└───────────────────────────┴───────────────────────────┘

*** DONE !!! ***
*** test_mae = 387.0574 hours = 16.1274 days


epoch,▁▁▂▂▃▃▄▄▅▅▅▅▆▆▇▇█
lr-Adam,███████▁
test_loss,▁
test_mae,▁
test_r2,▁
train_loss,█▁▁▁▁▁▁▁
train_mae,█▁▁▁▁▁▁▁
train_r2,▁███████
trainer/global_step,▁▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▇▇▇███
val_loss,▂▄▁▅█▃▅▃
+2,...


View run splendid-star-8 at: https://wandb.ai/huomi/zeleo/runs/m82slw63 View project at: https://wandb.ai/huomi/zeleo Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)

Find logs at: wandb/run-20260611_033840-m82slw63/logs